In [1]:
# @title 导入库


import dataclasses
import datetime
import functools
import math
import re
from typing import Optional

import cartopy.crs as ccrs
#from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
from IPython.display import HTML
import ipywidgets as widgets
import haiku as hk
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray

import os
os.environ["JAX_DISABLE_XLA"] = "1"



def parse_file_parts(file_name):
  return dict(part.split("-", 1) for part in file_name.split("_"))


In [1]:
import xarray as xr  
import pandas as pd
import numpy as np

# 设置数据文件存储的路径
dir_path_data = "/root/autodl-tmp/"
batch_size = 12  # 定义批次大小

# 使用 dask 进行懒加载，懒加载意味着数据不会一次性加载到内存中，而是按需加载
ds = xr.open_dataset(f"{dir_path_data}/example_batch_2011.nc", chunks={'time': 12})
ds

<xarray.Dataset>
Dimensions:   (batch: 30, time: 12, lat: 2041, lon: 4320, level: 3)
Coordinates:
  * batch     (batch) int64 0 1 2 3 4 5 6 7 8 9 ... 21 22 23 24 25 26 27 28 29
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 10 days 11 days
    datetime  (batch, time) datetime64[ns] dask.array<chunksize=(30, 12), meta=np.ndarray>
  * level     (level) float32 0.494 2.646 5.078
  * lat       (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * lon       (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
Data variables:
    siconc    (batch, time, lat, lon) float32 dask.array<chunksize=(30, 12, 2041, 4320), meta=np.ndarray>
    sithick   (batch, time, lat, lon) float32 dask.array<chunksize=(30, 12, 2041, 4320), meta=np.ndarray>
    so        (batch, time, level, lat, lon) float32 dask.array<chunksize=(30, 12, 3, 2041, 4320), meta=np.ndarray>
    thetao    (batch, time, level, lat, lon) float32 dask.array<chunksize=(30, 12, 3, 2041, 4320), meta=np.ndarray>
    uo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(30, 12, 3, 2041, 4320), meta=np.ndarray>
    usi       (batch, time, lat, lon) float32 dask.array<chunksize=(30, 12, 2041, 4320), meta=np.ndarray>
    vo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(30, 12, 3, 2041, 4320), meta=np.ndarray>
    vsi       (batch, time, lat, lon) float32 dask.array<chunksize=(30, 12, 2041, 4320), meta=np.ndarray>
    zos       (batch, time, lat, lon) float32 dask.array<chunksize=(30, 12, 2041, 4320), meta=np.ndarray>
Attributes:
    Conventions:               CF-1.4
    source:                    MERCATOR GLORYS12V1
    comment:                   CMEMS product
    title:                     daily mean fields from Global Ocean Physics An...
    history:                   2023/06/01 16:20:05 MERCATOR OCEAN Netcdf crea...
    institution:               MERCATOR OCEAN
    references:                http://www.mercator-ocean.fr
    copernicusmarine_version:  2.0.1

In [1]:
import xarray as xr  
import pandas as pd
import numpy as np

# 设置数据文件存储的路径
dir_path_data = "/root/autodl-tmp/"
batch_size = 12  # 定义批次大小

# 使用 dask 进行懒加载，懒加载意味着数据不会一次性加载到内存中，而是按需加载
ds = xr.open_dataset(f"{dir_path_data}/cmems_mod_glo_phy_my_0.083deg_P1D-m_multi-vars_180.00W-179.92E_80.00S-90.00N_0.49-5.08m_2019-01-01-2019-01-12.nc", 
                                          chunks={'time': 12})
                     # chunks={'time': 12, 'lat': 1024, 'lon': 1024, 'level': 5})

# 这里加载一个 NetCDF 文件，并将 time 维度分块成批次大小为 batch_size 的块

# 重命名维度，使得坐标轴的名称更符合用户需求
ds = ds.rename({'latitude': 'lat', 'longitude': 'lon', 'depth': 'level'})
# 将 `latitude` 维度重命名为 `lat`，`longitude` 重命名为 `lon`，`depth` 重命名为 `level`

# 将数据集的所有数据类型转换为 np.float32，以节省内存空间
ds = ds.astype(np.float32)

# 删除 level 维度中的第1层（假设这是要去除的一层深度数据）
ds = ds.isel(level=[0, 2, 4])
# ds = ds.drop_vars(['bottomT', 'mlotst'])

# 计算新的批次维度大小（时间轴上的批次数）
num_batches = ds.dims['time'] // batch_size
# `ds.dims['time']` 返回数据集在时间维度上的大小，除以批次大小，得到数据集总共有多少个批次

# 创建一个空列表，用于保存每个批次的数据
all_batches = []

# 按批次处理数据
for i in range(num_batches):
    start = i * batch_size  # 当前批次的起始时间索引
    end = start + batch_size  # 当前批次的结束时间索引
    batch_data = ds.isel(time=slice(start, end))  # 从数据集中提取时间范围内的数据
    all_batches.append(batch_data)  # 将当前批次添加到 all_batches 列表中

# 合并所有批次的数据集，沿时间维度拼接
example_batch = xr.concat(all_batches, dim='time')
# `xr.concat` 将 `all_batches` 列表中的各个数据集按时间维度拼接，得到一个完整的 batch 数据集

# 创建新的时间坐标数组
time_coords = example_batch.coords['time'].values  # 获取合并后的数据集的时间坐标
datetime_coords = np.full((num_batches, batch_size), np.datetime64('NaT'), dtype='datetime64[ns]')
# 创建一个空的 `datetime_coords` 数组，用于保存新的时间戳，初始为 NaT

# 按批次将时间坐标分配给新的时间坐标数组
for i in range(num_batches):
    start = i * batch_size  # 当前批次的起始时间索引
    end = (i + 1) * batch_size  # 当前批次的结束时间索引
    datetime_coords[i, :] = time_coords[start:end]  # 将每个批次对应的时间范围赋值给 `datetime_coords`
print("按批次将时间坐标分配给新的时间坐标数组")

# 在进行reshape时使用dask而非numpy
new_data_vars = {}
for var in example_batch.data_vars:
    data = example_batch[var].data  # 使用dask数组
    reshaped_data = data.reshape((num_batches, batch_size, *data.shape[1:]))
    new_data_vars[var] = (['batch', 'time'] + list(example_batch[var].dims[1:]), reshaped_data)
print("reshape")

# 创建数据集时避免将数据强制转换为numpy
example_batch = xr.Dataset(new_data_vars, coords={ 
    'level': example_batch['level'],  # 保留原来的 `level` 维度
    'lon': example_batch['lon'],  # 保留经度维度
    'lat': example_batch['lat'],  # 保留纬度维度
    'time': ('time', np.arange(batch_size)),  # 使用 0 到 batch_size-1 作为新的时间坐标
    'batch': ('batch', np.arange(num_batches)),  # 创建一个新的 `batch` 维度
    'datetime': (['batch', 'time'], datetime_coords),  # 新的时间坐标，按批次和时间维度存储
})
print("创建数据集")

# 将 time 转换为 timedelta64[ns] 类型
example_batch['time'] = pd.to_timedelta(example_batch['time'], unit='D')
# `pd.to_timedelta` 将整数时间坐标转换为 timedelta 格式，表示时间差

# 删除 `batch` 维度，保持最终数据集的维度只有 `time`, `level`, `lat`, `lon` 等
example_batch = example_batch.drop_vars('batch')

# 返回最终的数据集对象
example_batch

按批次将时间坐标分配给新的时间坐标数组
reshape
创建数据集


<xarray.Dataset>
Dimensions:   (batch: 1, time: 12, lat: 2041, lon: 4320, level: 3)
Coordinates:
  * level     (level) float32 0.494 2.646 5.078
  * lon       (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * lat       (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 10 days 11 days
    datetime  (batch, time) datetime64[ns] 2019-01-01 2019-01-02 ... 2019-01-12
Dimensions without coordinates: batch
Data variables:
    siconc    (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    sithick   (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    so        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    thetao    (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    uo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    usi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    vo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    vsi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    zos       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>

In [1]:
# import xarray as xr
# import pandas as pd
# import numpy as np

# dir_path_data = "/root/autodl-tmp/"
# batch_size = 12
# file_path = f"{dir_path_data}/cmems_mod_glo_phy_my_0.083deg_P1D-m_multi-vars_180.00W-179.92E_80.00S-90.00N_0.49-5.08m_2011-07-01-2011-12-30.nc"

# # 1) 读取时就启用 dask
# #    这里给一个比较稳妥的起点；如果内存更紧，可以把 512 改成 256
# ds = xr.open_dataset(
#     file_path,
#     chunks={"time": batch_size},
#     # chunks={"time": batch_size, "latitude": 512, "longitude": 512, "depth": 3},
# )

# # 2) 重命名维度
# ds = ds.rename({"latitude": "lat", "longitude": "lon", "depth": "level"})

# # 3) 只保留需要的 level
# ds = ds.isel(level=[0, 2, 4])

# # 4) 转 float32（只转数据变量）
# for var in ds.data_vars:
#     if ds[var].dtype != np.float32:
#         ds[var] = ds[var].astype(np.float32)

# # 5) 只保留能整除 batch_size 的时间长度
# num_batches = ds.sizes["time"] // batch_size
# valid_time = num_batches * batch_size
# ds = ds.isel(time=slice(0, valid_time))

# # 6) 一次性生成二维 datetime 坐标
# datetime_coords = ds.indexes["time"].values.reshape(num_batches, batch_size)

# # 7) 直接 reshape 各变量，不再先切 batch 再 concat
# new_data_vars = {}
# for var, da in ds.data_vars.items():
#     # 没有 time 维的变量，直接保留
#     if "time" not in da.dims:
#         new_data_vars[var] = da
#         continue

#     # 确保 time 在第一维，方便 reshape
#     if da.dims[0] != "time":
#         da = da.transpose("time", ...)

#     arr = da.data
#     reshaped = arr.reshape((num_batches, batch_size, *arr.shape[1:]))

#     new_data_vars[var] = (
#         ("batch", "time", *da.dims[1:]),
#         reshaped,
#         da.attrs,
#     )

# # 8) 重建 Dataset
# example_batch = xr.Dataset(
#     data_vars=new_data_vars,
#     coords={
#         "batch": np.arange(num_batches),
#         "time": pd.to_timedelta(np.arange(batch_size), unit="D"),
#         "datetime": (("batch", "time"), datetime_coords),
#         "level": ds["level"],
#         "lat": ds["lat"],
#         "lon": ds["lon"],
#     },
#     attrs=ds.attrs,
# )

# example_batch

/root/miniconda3/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3334: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explicitly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


<xarray.Dataset>
Dimensions:   (batch: 15, time: 12, lat: 2041, lon: 4320, level: 3)
Coordinates:
  * batch     (batch) int64 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 10 days 11 days
    datetime  (batch, time) datetime64[ns] 2011-07-01 2011-07-02 ... 2011-12-27
  * level     (level) float32 0.494 2.646 5.078
  * lat       (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * lon       (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
Data variables:
    siconc    (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    sithick   (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    so        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    thetao    (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    uo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    usi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    vo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    vsi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    zos       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
Attributes:
    title:                     daily mean fields from Global Ocean Physics An...
    comment:                   CMEMS product
    institution:               MERCATOR OCEAN
    references:                http://www.mercator-ocean.fr
    Conventions:               CF-1.4
    history:                   2023/06/01 16:20:05 MERCATOR OCEAN Netcdf crea...
    source:                    MERCATOR GLORYS12V1
    copernicusmarine_version:  2.0.1

In [2]:
# 保存最终的数据集到 NetCDF 文件
output_file_path = f"{dir_path_data}/eval_batch.nc"
example_batch = example_batch.chunk({
    'time' : 12
})# 保存路径
example_batch.to_netcdf(output_file_path,mode = 'w')  # 将数据保存为 NetCDF 格式文件
example_batch

<xarray.Dataset>
Dimensions:   (batch: 1, time: 12, lat: 2041, lon: 4320, level: 3)
Coordinates:
  * level     (level) float32 0.494 2.646 5.078
  * lon       (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * lat       (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 10 days 11 days
    datetime  (batch, time) datetime64[ns] dask.array<chunksize=(1, 12), meta=np.ndarray>
Dimensions without coordinates: batch
Data variables:
    siconc    (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    sithick   (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    so        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    thetao    (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    uo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    usi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    vo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    vsi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    zos       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>

In [ ]:
# import xarray as xr
# import pandas as pd
# import numpy as np

# dir_path_data = "/root/autodl-tmp/"

# # 读取 NetCDF 文件到 example_batch
# input_file_path = f"{dir_path_data}/example_batch_2010-0106.nc"
# example_batch0 = xr.open_dataset(input_file_path, chunks= {"batch": 1, "time": 12})
# # example_batch0 = xr.open_dataset(input_file_path, chunks= {"time": 12})

# input_file_path = f"{dir_path_data}/example_batch_2010-0712.nc"
# example_batch1 = xr.open_dataset(input_file_path, chunks= {"batch": 1, "time": 12})
# # example_batch1 = xr.open_dataset(input_file_path, chunks= {"time": 12})

# for var in example_batch0.data_vars:
#     try:
#         xr.concat([example_batch0[var], example_batch1[var]], dim="batch")
#     except Exception as e:
#         print(f"变量 {var} 出错: {e}")
#         print("example_batch0 dims:", example_batch0[var].dims)
#         print("example_batch1 dims:", example_batch1[var].dims)
#         print("example_batch0 coords:", list(example_batch0[var].coords))
#         print("example_batch1 coords:", list(example_batch1[var].coords))
#         break
        
        

# # 按 batch 维度进行拼接
# data_vars = {}
# for var in example_batch0.data_vars:
#     data_vars[var] = xr.concat([example_batch0[var], example_batch1[var]], dim="batch")

# # 构建拼接后的数据集，不直接使用 eval_batch.coords
# example_batch = xr.Dataset(data_vars)

# # # 创建 land_sea_mask 变量，避免中间副本
# # example_batch["land_sea_mask"] = (
# #     (~np.isnan(example_batch["so"].isel(time=0, level=0)))
# #     .astype(np.float32)
# #     .persist()  # 在 Dask 中持久化计算结果以减少重复计算
# # )

# # 直接保存到 NetCDF 文件，避免将所有数据加载到内存中
# output_file_path = f"{dir_path_data}/example_batch_2010.nc"
# example_batch.to_netcdf(output_file_path, engine="netcdf4", mode='w')

In [1]:
import xarray as xr
import numpy as np

dir_path_data = "/root/autodl-tmp/"

# 读取两个文件
example_batch0 = xr.open_dataset(
    f"{dir_path_data}/example_batch_2010.nc",
    chunks={"time": 12, "batch": 2}
)
example_batch1 = xr.open_dataset(
    f"{dir_path_data}/example_batch_2011.nc",
    chunks={"time": 12, "batch": 2}
)

# 给两个数据集都显式补上 batch 坐标
n0 = example_batch0.sizes["batch"]
n1 = example_batch1.sizes["batch"]

example_batch0 = example_batch0.assign_coords(
    batch=("batch", np.arange(n0))
)

example_batch1 = example_batch1.assign_coords(
    batch=("batch", np.arange(n0, n0 + n1))
)

# 直接拼整个 Dataset，不要逐变量拼
example_batch = xr.concat(
    [example_batch0, example_batch1],
    dim="batch",
    coords="minimal",
    compat="override"
)

# 保存
output_file_path = f"{dir_path_data}/example_batch.nc"
example_batch.to_netcdf(output_file_path, engine="netcdf4", mode="w")

In [2]:
example_batch

<xarray.Dataset>
Dimensions:   (lon: 4320, lat: 2041, time: 12, batch: 10, level: 17)
Coordinates:
  * lon       (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * lat       (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 10 days 11 days
    datetime  (batch, time) datetime64[ns] dask.array<chunksize=(1, 12), meta=np.ndarray>
  * level     (level) float32 0.494 2.646 5.078 7.93 ... 186.1 318.1 453.9 541.1
Dimensions without coordinates: batch
Data variables:
    siconc    (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    sithick   (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    so        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 17, 2041, 4320), meta=np.ndarray>
    thetao    (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 17, 2041, 4320), meta=np.ndarray>
    uo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 17, 2041, 4320), meta=np.ndarray>
    usi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    vo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 17, 2041, 4320), meta=np.ndarray>
    vsi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    zos       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>

In [1]:
import xarray as xr
import numpy as np

# 假设你的经纬度数据已经加载
input_file_path = "/root/autodl-tmp/example_batch_1-3.nc"
example_batch = xr.open_dataset(input_file_path, chunks={'level': 1, 'time': 1})


Yt = example_batch.diff(dim="time")

residual_scales = {}
for var in Yt.data_vars:
    residual_scales[var] = Yt[var].std(dim=["batch", "time", "lat", "lon"])
   
        
diffs_stddev_by_level = xr.Dataset(residual_scales)

# 检查目标目录是否存在并具有写入权限
import os
save_path = '/root/data/stats'
if not os.path.exists(save_path):
    os.makedirs(save_path)
    
try:
    diffs_stddev_by_level.to_netcdf(f'{save_path}/stats-diffs_stddev_by_level.nc', engine='scipy')
    print("标准化文件已保存")
except RuntimeError as e:
    print(f"保存标准化文件时出错: {e}")

example_batch

标准化文件已保存


<xarray.Dataset>
Dimensions:   (lon: 4320, lat: 2041, time: 12, batch: 10, level: 17)
Coordinates:
  * lon       (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * lat       (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 10 days 11 days
    datetime  (batch, time) datetime64[ns] dask.array<chunksize=(10, 1), meta=np.ndarray>
  * level     (level) float32 0.494 2.646 5.078 7.93 ... 186.1 318.1 453.9 541.1
Dimensions without coordinates: batch
Data variables:
    siconc    (batch, time, lat, lon) float32 dask.array<chunksize=(10, 1, 2041, 4320), meta=np.ndarray>
    sithick   (batch, time, lat, lon) float32 dask.array<chunksize=(10, 1, 2041, 4320), meta=np.ndarray>
    so        (batch, time, level, lat, lon) float32 dask.array<chunksize=(10, 1, 1, 2041, 4320), meta=np.ndarray>
    thetao    (batch, time, level, lat, lon) float32 dask.array<chunksize=(10, 1, 1, 2041, 4320), meta=np.ndarray>
    uo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(10, 1, 1, 2041, 4320), meta=np.ndarray>
    usi       (batch, time, lat, lon) float32 dask.array<chunksize=(10, 1, 2041, 4320), meta=np.ndarray>
    vo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(10, 1, 1, 2041, 4320), meta=np.ndarray>
    vsi       (batch, time, lat, lon) float32 dask.array<chunksize=(10, 1, 2041, 4320), meta=np.ndarray>
    zos       (batch, time, lat, lon) float32 dask.array<chunksize=(10, 1, 2041, 4320), meta=np.ndarray>

In [2]:
# @title 导入库


import dataclasses
import datetime
import functools
import math
import re
from typing import Optional

import cartopy.crs as ccrs
#from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
from IPython.display import HTML
import ipywidgets as widgets
import haiku as hk
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray

import os
os.environ["JAX_DISABLE_XLA"] = "1"



def parse_file_parts(file_name):
  return dict(part.split("-", 1) for part in file_name.split("_"))


In [3]:
# 创建一个新的变量 land_sea_mask
import xarray as xr
import numpy as np

# 假设你的经纬度数据已经加载
input_file_path = "/root/autodl-tmp/example_batch_1-3.nc"
# 使用 dask 延迟加载数据
example_batch = xr.open_dataset(input_file_path, chunks={'level': 1, 'time': 1, 'lat': 300, 'lon': 300, 'batch': 1})


# 创建 land_sea_mask 变量，避免中间副本
example_batch["land_sea_mask"] = (
    (~np.isnan(example_batch["so"].isel(time=0, level=0)))
    .astype(np.float32)
    .persist()  # 在 Dask 中持久化计算结果以减少重复计算
)

example_batch = example_batch.fillna(0)
example_batch = example_batch.astype(np.float32)
example_batch

<xarray.Dataset>
Dimensions:        (batch: 10, time: 12, lat: 2041, lon: 4320, level: 17)
Coordinates:
  * lon            (lon) float32 -180.0 -179.9 -179.8 ... 179.8 179.8 179.9
  * lat            (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time           (time) timedelta64[ns] 0 days 1 days ... 10 days 11 days
    datetime       (batch, time) datetime64[ns] dask.array<chunksize=(1, 1), meta=np.ndarray>
  * level          (level) float32 0.494 2.646 5.078 7.93 ... 318.1 453.9 541.1
Dimensions without coordinates: batch
Data variables:
    siconc         (batch, time, lat, lon) float32 dask.array<chunksize=(1, 1, 300, 300), meta=np.ndarray>
    sithick        (batch, time, lat, lon) float32 dask.array<chunksize=(1, 1, 300, 300), meta=np.ndarray>
    so             (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 1, 1, 300, 300), meta=np.ndarray>
    thetao         (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 1, 1, 300, 300), meta=np.ndarray>
    uo             (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 1, 1, 300, 300), meta=np.ndarray>
    usi            (batch, time, lat, lon) float32 dask.array<chunksize=(1, 1, 300, 300), meta=np.ndarray>
    vo             (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 1, 1, 300, 300), meta=np.ndarray>
    vsi            (batch, time, lat, lon) float32 dask.array<chunksize=(1, 1, 300, 300), meta=np.ndarray>
    zos            (batch, time, lat, lon) float32 dask.array<chunksize=(1, 1, 300, 300), meta=np.ndarray>
    land_sea_mask  (batch, lat, lon) float32 dask.array<chunksize=(1, 300, 300), meta=np.ndarray>

In [4]:
import xarray as xr
import numpy as np

# 常量
SEC_PER_DAY = 86400
_AVG_DAY_PER_YEAR = 365.25

# 计算进度
seconds_since_epoch = (example_batch['time'].astype('int64') // 10**9).values
longitude = example_batch['lon'].values  # 经度

year_progress = data_utils.get_year_progress(seconds_since_epoch)
day_progress = data_utils.get_day_progress(seconds_since_epoch, longitude)

# 生成进度特征
year_features = data_utils.featurize_progress('year_progress', ['time'], year_progress)
day_features = data_utils.featurize_progress('day_progress', ['time', 'lon'], day_progress)

example_batch = example_batch.assign(year_progress_sin=year_features['year_progress_sin'],
                               year_progress_cos=year_features['year_progress_cos'],
                               day_progress_sin=day_features['day_progress_sin'],
                               day_progress_cos=day_features['day_progress_cos'])

# 创建掩码，排除数值为0的部分
masked_example_batch = example_batch.where(example_batch != 0)

# 计算每一天的全图均值和标准差
daily_mean = masked_example_batch.mean(dim=['lat', 'lon','time','batch'], skipna=True)
daily_stddev = masked_example_batch.std(dim=['lat', 'lon','time','batch'], skipna=True)

save_path = '/root/data/stats'
# 分别保存均值、标准差和标准化值到三个独立的NetCDF文件
try:
    daily_mean.to_netcdf(f'{save_path}/stats-mean_by_level.nc', engine='scipy')
    print("均值文件已保存")
except RuntimeError as e:
    print(f"保存均值文件时出错: {e}")

try:
    daily_stddev.to_netcdf(f'{save_path}/stats-stddev_by_level.nc', engine='scipy')
    print("标准差文件已保存")
except RuntimeError as e:
    print(f"保存标准差文件时出错: {e}")


    
# 去除进度特征
example_batch = example_batch.drop_vars([
    'year_progress_sin', 'year_progress_cos',
    'day_progress_sin', 'day_progress_cos'
])
example_batch

均值文件已保存
标准差文件已保存


<xarray.Dataset>
Dimensions:        (batch: 10, time: 12, lat: 2041, lon: 4320, level: 17)
Coordinates:
  * lon            (lon) float32 -180.0 -179.9 -179.8 ... 179.8 179.8 179.9
  * lat            (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time           (time) timedelta64[ns] 0 days 1 days ... 10 days 11 days
    datetime       (batch, time) datetime64[ns] dask.array<chunksize=(1, 1), meta=np.ndarray>
  * level          (level) float32 0.494 2.646 5.078 7.93 ... 318.1 453.9 541.1
Dimensions without coordinates: batch
Data variables:
    siconc         (batch, time, lat, lon) float32 dask.array<chunksize=(1, 1, 300, 300), meta=np.ndarray>
    sithick        (batch, time, lat, lon) float32 dask.array<chunksize=(1, 1, 300, 300), meta=np.ndarray>
    so             (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 1, 1, 300, 300), meta=np.ndarray>
    thetao         (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 1, 1, 300, 300), meta=np.ndarray>
    uo             (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 1, 1, 300, 300), meta=np.ndarray>
    usi            (batch, time, lat, lon) float32 dask.array<chunksize=(1, 1, 300, 300), meta=np.ndarray>
    vo             (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 1, 1, 300, 300), meta=np.ndarray>
    vsi            (batch, time, lat, lon) float32 dask.array<chunksize=(1, 1, 300, 300), meta=np.ndarray>
    zos            (batch, time, lat, lon) float32 dask.array<chunksize=(1, 1, 300, 300), meta=np.ndarray>
    land_sea_mask  (batch, lat, lon) float32 dask.array<chunksize=(1, 300, 300), meta=np.ndarray>

In [ ]:
# @title 加载规范化数据
# Rewrite by S.F. Sune, https://github.com/sfsun67.
# 定义了一个变量dir_path_stats，它存储了数据统计文件所在的目录路径。
import xarray
import os

dir_path_stats = "/root/data/stats/"

# 这行代码使用with语句和open函数以二进制读取模式（“rb”）打开一个名为stats-diffs_stddev_by_level.nc的文件。
# with open(f"{dir_path_stats}/stats-diffs_stddev_by_level.nc", "rb") as f:
#   diffs_stddev_by_level = xarray.load_dataset(f).compute()
# 类似于第4行，这行代码打开了另一个文件stats-mean_by_level.nc。
with open(f"{dir_path_stats}/stats-mean_by_level.nc", "rb") as f:
  mean_by_level = xarray.load_dataset(f).compute()
# 再次类似于第4行，这行代码打开了第三个文件stats-stddev_by_level.nc。
with open(f"{dir_path_stats}/stats-stddev_by_level.nc", "rb") as f:
  stddev_by_level = xarray.load_dataset(f).compute()
# 这行代码使用with语句和open函数以二进制读取模式（“rb”）打开一个名为stats-diffs_stddev_by_level.nc的文件。
with open(f"{dir_path_stats}/stats-diffs_stddev_by_level.nc", "rb") as f:
  diffs_stddev_by_level = xarray.load_dataset(f).compute()

In [6]:
diffs_stddev_by_level

<xarray.Dataset>
Dimensions:  (level: 3)
Coordinates:
  * level    (level) float32 0.494 2.646 5.078
Data variables:
    so       (level) float32 0.08962 0.07599 0.06856
    thetao   (level) float32 0.186 0.1648 0.1586
    uo       (level) float32 0.06233 0.06015 0.05855
    vo       (level) float32 0.06598 0.06285 0.06068
    siconc   float32 0.05462
    sithick  float32 0.05354
    vsi      float32 0.07438
    zos      float32 0.02339
    usi      float32 0.0692

In [7]:
mean_by_level

<xarray.Dataset>
Dimensions:            (level: 3)
Coordinates:
  * level              (level) float32 0.494 2.646 5.078
Data variables: (12/14)
    so                 (level) float32 34.04 34.05 34.06
    thetao             (level) float32 14.18 14.15 14.14
    uo                 (level) float32 -0.003989 -0.002443 -0.001364
    vo                 (level) float32 0.004731 0.005365 0.005792
    siconc             float32 0.8556
    sithick            float32 1.221
    ...                 ...
    usi                float32 -0.01344
    land_sea_mask      float32 1.0
    year_progress_sin  float32 0.1029
    year_progress_cos  float32 0.9938
    day_progress_sin   float32 2.355e-09
    day_progress_cos   float32 6.063e-08